# Notebook_C (dùng chung, phần dữ liệu, Sinh viên A): thu thập, so sánh nguồn, làm sạch, EDA và SQL

**Dùng chung cho mọi đề tài của môn.** Chỉ cần điền khối CONFIG bên dưới (tên đề tài, nguồn, tên cột, tần suất) rồi chạy tuần tự. Mọi ô code có chú thích tiếng Anh; phần giải thích, lý do và cách đọc kết quả bằng tiếng Việt nằm ngay trên mỗi ô.

**Sản phẩm bàn giao cho Sinh viên B**: `data/processed/feat.parquet`, `data/processed/manifest.json`, thư mục `report/` (bảng RQ1, hình, nhật ký làm sạch, AI Audit Log).

**Ba câu hỏi nghiên cứu chung (áp dụng cho mọi đề, mỗi nhóm thêm 1-2 câu riêng theo đề):**

* **RQ1 (dữ liệu và mô tả, sâu hơn)**: Hai nguồn dữ liệu (chính và phụ) có nhất quán với nhau không, biến mục tiêu biến động theo mùa, theo đơn vị và theo biến phụ như thế nào, và phần nào của biến động được giải thích bằng lịch so với bằng nguồn phụ?
* **RQ2 (mô hình và tổng quát hoá)**: Trong sáu họ mô hình (mốc naive, ridge, random forest, LightGBM toàn cục, tuyến tính dài hạn, mô hình nền zero-shot), mô hình nào tốt nhất theo tầm dự báo, và kết quả có giữ được khi lùi gốc dự báo theo thời gian và khi gặp đơn vị chưa thấy không?
* **RQ3 (tin cậy và vận hành)**: Khoảng tin cậy có đúng bao phủ không, cảnh báo sự kiện (mục tiêu vượt phân vị cao) đạt recall và precision bao nhiêu, sai số thay đổi thế nào khi chế độ đổi, và nguồn phụ đóng góp bao nhiêu (ablation)?

Notebook_C trả lời RQ1 và chuẩn bị dữ liệu cho RQ2, RQ3 (Notebook_D).

In [1]:
# CONFIG: fill in once for your group, both notebooks read the same block
TOPIC = "Weekly Street Tree Maintenance Demand Forecasting for SF Neighbourhoods"
GROUP = "Nhom 10"
PRIMARY_SOURCE = {"name": "SF 311 Cases (Tree Maintenance)",
                   "url": "https://data.sfgov.org/City-Infrastructure/311-Cases/vw6y-z8j6",
                   "license": "Public Domain Dedication and License (PDDL) v1.0",
                   "path": "https://data.sfgov.org/resource/vw6y-z8j6.csv?$where=service_name=%27Tree%20Maintenance%27&$limit=500000"}
SECOND_SOURCE = {"name": "SF Street-Use Permits",
                  "url": "https://data.sf.gov/City-Infrastructure/Street-Use-Permits/b6tj-gt35",
                  "license": "Public Domain Dedication and License (PDDL) v1.0",
                  "path": "https://data.sfgov.org/resource/b6tj-gt35.csv?$where=permit_start_date>=%272015-01-01%27&$limit=500000"}
THIRD_SOURCE = {"name": "NOAA GHCN-Daily Weather (SFO Airport station)",
                 "url": "https://www.ncei.noaa.gov/access/search/data-search/daily-summaries",
                 "license": "U.S. Public Domain (NOAA/NCEI)",
                 "path": "https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries&dataTypes=TMAX,TMIN,PRCP,AWND&stations=USW00023234&startDate=2015-01-01&endDate=2026-09-19&format=csv&units=metric"}
UNIT_COL   = "nbhd"
TIME_COL   = "requested_datetime"
TARGET_COL = "n"
EXOG_COLS  = ["permits_active"]
FREQ       = "D"
HORIZONS   = [1, 7]
SEASON     = 7
TEST_START = "2025-01-01"
EVENT_QUANTILE = 0.95
print("Topic:", TOPIC); print("Group:", GROUP)

Topic: Weekly Street Tree Maintenance Demand Forecasting for SF Neighbourhoods
Group: Nhom 10


In [2]:
# AI Audit Log helper: every prompt that changed your work is one row (2 minutes per entry, 3-5 per week)
import pandas as pd, os, datetime as dt
os.makedirs("report", exist_ok=True)
AUDIT_PATH = "report/ai_audit_log.csv"
def audit(step, prompt, tool, ai_output_summary, verified_how, decision, hallucination=False):
    """Append one entry. decision: what you kept / changed / rejected. hallucination=True if the AI answer was wrong and you caught it."""
    row = {"date": dt.date.today().isoformat(), "group": GROUP, "step": step, "prompt": prompt[:500], "tool": tool,
           "ai_output": ai_output_summary[:500], "verified_how": verified_how[:300], "decision": decision[:300], "hallucination": int(hallucination)}
    df = pd.DataFrame([row])
    df.to_csv(AUDIT_PATH, mode="a", header=not os.path.exists(AUDIT_PATH), index=False)
    print("audit entry saved:", step)
# example (delete after reading): audit("Step 2", "Given columns ... how to treat gaps longer than 3 steps?", "Claude", "suggested interpolate(limit=3) then drop", "checked share of gaps > 3 in Q4 below", "kept limit=3, dropped 1.2% rows")

## Hướng dẫn hỏi AI (điền đề tài của nhóm vào chỗ trống)

Quy tắc: hỏi **cụ thể** (kèm tên cột, kích thước, thông báo lỗi), yêu cầu AI **giải thích lý do** và **nêu cách kiểm tra**, rồi tự kiểm tra trước khi dùng. Mỗi prompt làm thay đổi bài phải ghi vào AI Audit Log bằng hàm `audit(...)` ở ô trên. Mẫu prompt theo bước (thay `{TOPIC}`, `{cột}` bằng thông tin thật của nhóm):

| Bước | Mẫu prompt | Cần tự kiểm tra gì |
|---|---|---|
| Thu thập | "Đề tài của tôi là {TOPIC}. Nguồn chính là {PRIMARY_SOURCE}. Hãy gợi ý 3 nguồn phụ công khai (thời tiết, sự kiện, giá, lịch) có thể ghép theo cột {TIME_COL} và {UNIT_COL}, kèm URL tải và giấy phép." | Mở URL, xác nhận có tải được và giấy phép cho phép dùng |
| So sánh nguồn | "Tôi có hai nguồn về cùng biến {TARGET_COL} ở tần suất {FREQ}. Hãy đề xuất 4 chỉ số để so sánh độ phủ, độ trễ cập nhật và độ lệch giữa hai nguồn." | Chạy lại số trên dữ liệu thật, không dùng số AI đưa |
| Làm sạch | "Cột {cột} có {x}% thiếu, thiếu theo cụm dài nhất {n} bước. Nên nội suy hay bỏ? Giải thích rủi ro rò rỉ tương lai." | Kiểm tra nội suy không dùng giá trị tương lai |
| SQL | "Viết truy vấn DuckDB tạo đặc trưng trễ {SEASON} bước và trung bình trượt theo {UNIT_COL}, có WINDOW, không rò rỉ." | Đếm dòng, kiểm tra cột NULL ở đầu chuỗi |
| EDA | "Từ bảng thống kê sau (dán bảng), nêu 3 nhận xét có thể kiểm chứng và 2 điều cần cảnh giác." | Mỗi nhận xét phải chỉ ra được ô số liệu tương ứng |
| Lỗi | "Lỗi: {dán nguyên văn}. Ngữ cảnh: {ô code}. Nguyên nhân và cách sửa tối thiểu?" | Sửa xong chạy lại ô test |

Ví dụ ghi Audit Log sau khi hỏi: `audit("Thu thập", "Đề tài ... gợi ý nguồn phụ", "Claude", "3 nguồn: NOAA ISD, ...", "mở 3 URL, 1 URL lỗi 404", "dùng NOAA ISD, loại nguồn 404", hallucination=True)`.

## Bước 0: môi trường và cấu trúc thư mục

**Làm gì**: tạo môi trường ảo một lần, cài thư viện, tạo thư mục chuẩn. **Vì sao**: cả hai sinh viên và giảng viên chạy cùng phiên bản thư viện thì kết quả tái tạo được; `requirements.txt` là bằng chứng.

**Cấu trúc**: `data/raw` (file gốc, không sửa), `data/processed` (parquet sạch), `sql/` (truy vấn), `report/` (bảng, hình, nhật ký), `notebooks/`.

In [3]:
# Step 0a: run once in a terminal (not in the notebook)
# python -m venv .venv && source .venv/bin/activate      (Windows: .venv\Scripts\activate)
# pip install pandas numpy duckdb pyarrow scikit-learn lightgbm statsmodels matplotlib seaborn ydata-profiling mapie shap requests
# pip freeze > requirements.txt
import os
for d in ["data/raw", "data/processed", "sql", "report", "notebooks"]: os.makedirs(d, exist_ok=True)
print("folders ready")

folders ready


In [4]:
# Step 0b: imports and plotting style (English labels in figures, no top/right spines, no in-figure titles)
import pandas as pd, numpy as np, duckdb, json, hashlib, warnings, re
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore"); np.random.seed(42)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
TEAL, ACC, GREY = "#1B6B6D", "#F4A261", "#9FBFBF"
def style(ax): ax.spines[["top", "right"]].set_visible(False)
def savefig(fig, name): fig.tight_layout(); fig.savefig(f"report/{name}.png", dpi=300); plt.close(fig); print("saved report/" + name + ".png")
con = duckdb.connect()
print("pandas", pd.__version__, "duckdb", duckdb.__version__)

pandas 2.3.3 duckdb 1.5.5


## Bước 1: bài toán, nguồn dữ liệu và kế hoạch thu thập

**Làm gì**: viết vấn đề bằng 5 câu, đăng ký hai nguồn (chính và phụ) vào sổ nguồn (`report/data_sources.csv`) với URL, giấy phép, ngày tải, số dòng. **Vì sao**: phần Data của bài báo cần ghi rõ nguồn là công trình nào, năm nào, số dòng, số cột và lý do chọn; sổ nguồn giúp viết phần đó trong hai phút.

**Nguồn phụ nên chọn**: biến có cơ chế ảnh hưởng rõ đến mục tiêu (thời tiết với nhu cầu, giá với sạc, sự kiện với lưu lượng), có cùng hoặc cao hơn tần suất mục tiêu, và có thể ghép theo thời gian và đơn vị.

In [5]:
# Step 1a: source registry (append every file you download; this becomes Table "Data sources" in the paper)
SOURCES_PATH = "report/data_sources.csv"
def register_source(src, rows=None, cols=None, start=None, end=None, note=""):
    """src: dict with name, url, license, path. rows/cols/start/end are filled after loading."""
    row = {**src, "downloaded": pd.Timestamp.today().date().isoformat(), "rows": rows, "cols": cols, "start": start, "end": end, "note": note}
    df = pd.DataFrame([row]); df.to_csv(SOURCES_PATH, mode="a", header=not os.path.exists(SOURCES_PATH), index=False)
    return row
print("registry at", SOURCES_PATH)

registry at report/data_sources.csv


In [6]:
# Step 1b: load the primary source (edit the reader to match your file: csv, parquet, or an API export saved to data/raw)
def load_table(path):
    """Read csv/parquet with DuckDB so large files never need to fit in memory; returns a pandas DataFrame."""
    if path.endswith(".paSrquet"): return con.execute(f"SELECT * FROM read_parquet('{path}')").df()
    return con.execute(f"SELECT * FROM read_csv_auto('{path}', union_by_name=true, sample_size=-1)").df()
raw = load_table(PRIMARY_SOURCE["path"])
raw.columns = [c.strip() for c in raw.columns]
raw["nbhd"] = raw["analysis_neighborhood"]
raw[TIME_COL] = pd.to_datetime(raw[TIME_COL], errors="coerce", utc=False)
print(raw.shape); display(raw.head(3)); display(raw.dtypes.value_counts())
register_source(PRIMARY_SOURCE, rows=len(raw), cols=raw.shape[1], start=str(raw[TIME_COL].min()), end=str(raw[TIME_COL].max()))

(175277, 26)


,service_request_id,requested_datetime,closed_date,updated_datetime,status_description,status_notes,agency_responsible,service_name,service_subtype,service_details,address,street,supervisor_district,neighborhoods_sffind_boundaries,analysis_neighborhood,police_district,lat,long,point,point_geom,source,media_url,bos_2012,data_as_of,data_loaded_at,nbhd
0,13384540,2021-01-19 06:52:00,2021-01-19 12:51:29,2021-01-19 12:51:29,Closed,Case Resolved - Issue addressed by staff.,RPD Park Service Area 1 Queue,Tree Maintenance,Trees - Damaged_Tree,Fallen_tree,"410 FUNSTON AVE, SAN FRANCISCO, CA, 94118",FUNSTON AVE,1.0,Inner Richmond,Inner Richmond,RICHMOND,37.780357,-122.471596,"\n, \n(37.78035736, -122.47159576)",POINT (-122.471595764 37.780357361),Phone,None,1.0,2024-02-20 10:35:55,2024-09-11 17:09:50,Inner Richmond
1,13451918,2021-02-04 10:36:00,2021-02-08 16:36:43,2021-02-08 16:36:43,Closed,Case Resolved - topping for signs is illegal t...,DPW Ops Queue,Tree Maintenance,Trees - Overgrown_Tree,Pruning_request,"3001 GEARY BLVD, SAN FRANCISCO, CA, 94118",GEARY BLVD,1.0,Lone Mountain,Lone Mountain/USF,RICHMOND,37.781761,-122.451263,"\n, \n(37.78176117, -122.45126343)",POINT (-122.451263428 37.781761169),Phone,None,1.0,2024-02-20 10:35:55,2024-09-11 17:09:50,Lone Mountain/USF
2,13402143,2021-01-22 18:00:00,2021-04-05 13:47:12,2021-04-05 13:47:12,Closed,Case is a Duplicate,DPW Ops Queue,Tree Maintenance,Trees - Tree_Other,Other,"324 LOMBARD ST, SAN FRANCISCO, CA, 94133",LOMBARD ST,3.0,Telegraph Hill,North Beach,CENTRAL,37.803572,-122.406997,"\n, \n(37.80357167, -122.40699667)",POINT (-122.40699667 37.80357167),Mobile/Open311,http://mobile311.sfgov.org/reports/13402143/ph...,3.0,2024-02-20 10:35:55,2024-09-11 17:09:50,North Beach


object            16
datetime64[us]     5
float64            4
int64              1
Name: count, dtype: int64

{'name': 'SF 311 Cases (Tree Maintenance)',
 'url': 'https://data.sfgov.org/City-Infrastructure/311-Cases/vw6y-z8j6',
 'license': 'Public Domain Dedication and License (PDDL) v1.0',
 'path': 'https://data.sfgov.org/resource/vw6y-z8j6.csv?$where=service_name=%27Tree%20Maintenance%27&$limit=500000',
 'downloaded': '2026-09-19',
 'rows': 175277,
 'cols': 26,
 'start': '2008-07-01 01:56:58',
 'end': '2026-09-17 22:00:58',
 'note': ''}

In [7]:
# Step 1c: load the secondary source (covariates); it must share TIME_COL (and UNIT_COL if it is unit-specific)
sec = load_table(SECOND_SOURCE["path"]); sec.columns = [c.strip() for c in sec.columns]
sec = sec.rename(columns={"permit_start_date": TIME_COL, "analysis_neighborhood": UNIT_COL})
sec[TIME_COL] = pd.to_datetime(sec[TIME_COL], errors="coerce")
SEC_HAS_UNIT = UNIT_COL in sec.columns
print(sec.shape, "unit-specific:", SEC_HAS_UNIT); display(sec.head(3))
register_source(SECOND_SOURCE, rows=len(sec), cols=sec.shape[1], start=str(sec[TIME_COL].min()), end=str(sec[TIME_COL].max()))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(446240, 31) unit-specific: True


,permit_number,cnn,unique_identifier,streetname,cross_street_1,cross_street_2,permit_type,agent,agentphone,permit_purpose,approved_date,status,permit_zipcode,requested_datetime,permit_end_date,permit_address,contact,inspector,curbrampwork,planchecker,x,y,latitude,longitude,bpa,location,the_geom,nbhd,supervisor_district,data_as_of,data_loaded_at
0,26N-00104,9100000,26N-00104_9100000,MISSION ST,06TH ST,07TH ST,NightNoise,Pacific Gas & Electric,5102920681,47305815 - 1045 Mission St. - FIG - gas service,2026-03-24 09:05:21,EXPIRED,94103,2026-03-31 00:00:00,2026-04-01 00:00:00,None,800-pge-5000,None,<NA>,Ollie Sears,6.009763e+06,2.112142e+06,37.780091,-122.409823,None,"\n, \n(37.780091425877835, -122.4098226369408)",POINT (-122.409822637 37.780091426),South of Market,6,2026-04-03 02:58:31,2026-04-04 05:22:57
1,26TOC-01396,1896000,26TOC-01396_1896000,40TH AVE,QUINTARA ST,RIVERA ST,TempOccup,Pacific Gas & Electric,5102920681,None,2026-04-02 08:41:25,CLOSED,94116,2026-04-12 10:00:00,2026-04-12 18:00:00,None,800-pge-5000,None,<NA>,None,5.983876e+06,2.100529e+06,37.746719,-122.498518,None,"\n, \n(37.7467190572965, -122.49851796866787)",POINT (-122.498517969 37.746719057),Sunset/Parkside,4,2026-04-14 03:17:13,2026-04-15 05:24:06
2,26EXC-01641,408000,26EXC-01641_408000,08TH ST,NATOMA ST,HOWARD ST,Excavation,Ronan Construction Inc.,415-779-5262,"8th St, Clay St and Leavenworth St - Pavement ...",2026-04-15 17:00:43,APPROVED,94103,2026-05-01 00:00:00,2026-10-28 00:00:00,None,415-779-5262,None,<NA>,Noah Wong,6.009141e+06,2.110826e+06,37.776441,-122.411882,None,"\n, \n(37.77644115815177, -122.41188203249555)",POINT (-122.411882032 37.776441158),South of Market,6,2026-04-16 03:34:45,2026-04-17 05:23:32


{'name': 'SF Street-Use Permits',
 'url': 'https://data.sf.gov/City-Infrastructure/Street-Use-Permits/b6tj-gt35',
 'license': 'Public Domain Dedication and License (PDDL) v1.0',
 'path': 'https://data.sfgov.org/resource/b6tj-gt35.csv?$where=permit_start_date>=%272015-01-01%27&$limit=500000',
 'downloaded': '2026-09-19',
 'rows': 446240,
 'cols': 31,
 'start': '2015-01-01 00:00:00',
 'end': '2209-09-27 23:00:00',
 'note': ''}

### Xuất và đọc lại dữ liệu bằng CSV

**Quy ước file**: mọi bước lưu một file CSV riêng để giảng viên và bạn cùng nhóm mở được bằng Excel mà không cần Python: `data/raw/primary_snapshot.csv` (bản gốc đã chuẩn tên cột và kiểu thời gian, không sửa giá trị), `data/processed/clean.csv` (sau làm sạch, một dòng một đơn vị-thời điểm), `data/processed/feat.csv` (bảng đặc trưng bàn giao). Parquet vẫn lưu song song vì nhanh và giữ đúng kiểu dữ liệu; CSV là bản để đọc và nộp. Khi đọc lại CSV, luôn ép kiểu thời gian bằng `parse_dates` và kiểm tra số dòng trùng khớp với manifest.

In [8]:
# Export the standardised raw snapshot to CSV and read it back to confirm the round trip (dtypes and row count)
RAW_CSV = "data/raw/primary_snapshot.csv"
raw.to_csv(RAW_CSV, index=False)
check = pd.read_csv(RAW_CSV, parse_dates=[TIME_COL])
assert len(check) == len(raw), "row count changed in the CSV round trip"
print("wrote", RAW_CSV, "| rows", len(check), "| columns", list(check.columns)[:8])

wrote data/raw/primary_snapshot.csv | rows 175277 | columns ['service_request_id', 'requested_datetime', 'closed_date', 'updated_datetime', 'status_description', 'status_notes', 'agency_responsible', 'service_name']


### Bước 1d: thu thập thêm dữ liệu (mở rộng phạm vi thời gian hoặc đơn vị)

**Làm gì**: nếu nguồn cho phép, tải thêm giai đoạn hoặc thêm đơn vị (trạm, vùng) để bảng đủ khoảng 100 nghìn dòng và có đủ ít nhất hai mùa hoặc hai chế độ. Hàm dưới ghép nhiều file tải theo năm hoặc theo đơn vị và loại trùng. **Vì sao**: mô hình toàn cục và kiểm thử đơn vị chưa thấy cần nhiều đơn vị; đổi chế độ chỉ đo được khi có dữ liệu trước và sau.

In [9]:
# Step 1d: combine several downloaded chunks (data/raw/primary_*.csv) into one table, deduplicated on unit and time
import glob
chunks = sorted(glob.glob(PRIMARY_SOURCE["path"].replace(".csv", "_*.csv")))
if chunks:
    extra = pd.concat([load_table(f) for f in chunks]); extra[TIME_COL] = pd.to_datetime(extra[TIME_COL], errors="coerce")
    before = len(raw); raw = pd.concat([raw, extra]).drop_duplicates([UNIT_COL, TIME_COL] if UNIT_COL in raw.columns else [TIME_COL])
    print(f"added {len(raw) - before} rows from {len(chunks)} chunks")
else:
    print("no extra chunks found (pattern primary_*.csv); skip if your source is a single file")
if UNIT_COL not in raw.columns: raw[UNIT_COL] = "all"; print("single-series data: UNIT_COL set to 'all'")

no extra chunks found (pattern primary_*.csv); skip if your source is a single file


## Bước 2: so sánh hai nguồn và làm sạch có nhật ký

**So sánh nguồn (RQ1a)**: trước khi ghép, đo (1) khoảng thời gian chồng nhau, (2) tỷ lệ bước thời gian có ở nguồn này mà không có ở nguồn kia, (3) nếu hai nguồn có cùng một biến thì tương quan và độ lệch trung bình, (4) độ trễ cập nhật (thời điểm cuối của mỗi nguồn). Bảng này vào phần Data của bài.

**Làm sạch có nhật ký**: mỗi thao tác ghi số dòng trước, sau và lý do; bảng nhật ký là bằng chứng để người đọc tin dữ liệu.

In [10]:
# Step 2a: source comparison table
def compare_sources(a, b, time_col, common_var=None):
    ta, tb = a[time_col].dropna(), b[time_col].dropna()
    ov_start, ov_end = max(ta.min(), tb.min()), min(ta.max(), tb.max())
    ga = set(ta.dt.floor(FREQ).unique()); gb = set(tb.dt.floor(FREQ).unique())
    rows = [["overlap start", ov_start], ["overlap end", ov_end], ["steps only in primary", len(ga - gb)], ["steps only in secondary", len(gb - ga)],
            ["latest timestamp primary", ta.max()], ["latest timestamp secondary", tb.max()]]
    if common_var and common_var in a.columns and common_var in b.columns:
        m = a.groupby(time_col)[common_var].mean().to_frame("a").join(b.groupby(time_col)[common_var].mean().to_frame("b"), how="inner")
        rows += [["correlation of common variable", round(m.a.corr(m.b), 3)], ["mean difference (primary minus secondary)", round((m.a - m.b).mean(), 3)]]
    return pd.DataFrame(rows, columns=["metric", "value"])
cmp_tbl = compare_sources(raw, sec, TIME_COL, common_var=None)     # set common_var="temp" if both sources carry the same variable
cmp_tbl.to_csv("report/table_source_comparison.csv", index=False); display(cmp_tbl)

,metric,value
0,overlap start,2015-01-01 00:00:00
1,overlap end,2026-09-17 22:00:58
2,steps only in primary,2381
3,steps only in secondary,78
4,latest timestamp primary,2026-09-17 22:00:58
5,latest timestamp secondary,2209-09-27 23:00:00


**Nhận xét về mức nhất quán giữa hai nguồn**: hai nguồn chồng lấp 11.7 năm (2015-01-01 đến
2026-09-17), trong vùng chồng lấp chỉ có 78 ngày (0.9%) chỉ xuất hiện ở nguồn phụ — mức nhất
quán tốt. 2381 ngày chỉ có ở nguồn chính là hợp lý vì nguồn chính (311 cases) bắt đầu từ 2008
còn nguồn phụ (permits) chỉ tải từ 2015, không phải lỗi ghép dữ liệu. Bảng so sánh còn giúp
phát hiện một lỗi nhập liệu thật: timestamp mới nhất của nguồn phụ hiện ra là năm 2209 (rõ ràng
sai), đã được lọc bỏ ở bước làm sạch ngay sau đó.

In [11]:
# Step 2b: cleaning with a log; each step records rows before and after and the reason
clean_log = []
def step(df, name, fn, reason):
    n0 = len(df); out = fn(df); clean_log.append([name, n0, len(out), n0 - len(out), reason]); return out
df = (raw.dropna(subset=[TIME_COL])
         .assign(**{TIME_COL: lambda x: x[TIME_COL].dt.floor(FREQ)})
         .groupby([UNIT_COL, TIME_COL]).size().rename(TARGET_COL).reset_index())
full_idx = pd.MultiIndex.from_product(
    [df[UNIT_COL].unique(), pd.date_range(df[TIME_COL].min(), df[TIME_COL].max(), freq=FREQ)],
    names=[UNIT_COL, TIME_COL])
df = df.set_index([UNIT_COL, TIME_COL]).reindex(full_idx, fill_value=0)[TARGET_COL].reset_index()

sec = step(sec, "drop unrealistic future dates (secondary)",
           lambda d: d[d[TIME_COL] <= pd.Timestamp.now() + pd.Timedelta(days=365)],
           "permit_start_date has data-entry errors far in the future (e.g. year 2209)")
df = step(df, "parse time", lambda x: x.dropna(subset=[TIME_COL]), "rows with unparseable timestamps")
df = step(df, "drop duplicates", lambda x: x.drop_duplicates([UNIT_COL, TIME_COL]), "same unit and timestamp")
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = step(df, "drop missing target", lambda x: x.dropna(subset=[TARGET_COL]), "target missing")
lo, hi = df[TARGET_COL].quantile([0.001, 0.999])
df = step(df, "physical range", lambda x: x[(x[TARGET_COL] >= lo) & (x[TARGET_COL] <= hi)], f"outside [{lo:.3g}, {hi:.3g}] (0.1% tails; replace with a physical range if you know it)")
pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]).to_csv("report/table_cleaning_log.csv", index=False)
display(pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]))

,step,rows_before,rows_after,rows_removed,reason
0,drop unrealistic future dates (secondary),446240,446232,8,permit_start_date has data-entry errors far in...
1,parse time,272773,272773,0,rows with unparseable timestamps
2,drop duplicates,272773,272773,0,same unit and timestamp
3,drop missing target,272773,272773,0,target missing
4,physical range,272773,272501,272,"outside [0, 13] (0.1% tails; replace with a ph..."


In [12]:
# Step 2c: regular time grid per unit, then short-gap interpolation (limit = 3 steps, never across long gaps)
df = df.set_index(TIME_COL).groupby(UNIT_COL)[[TARGET_COL] + [c for c in df.columns if c not in (UNIT_COL, TIME_COL, TARGET_COL) and pd.api.types.is_numeric_dtype(df[c])]].resample(FREQ).mean().reset_index()
gap = df.groupby(UNIT_COL)[TARGET_COL].apply(lambda s: s.isna().mean()).rename("missing_share")
df[TARGET_COL] = df.groupby(UNIT_COL)[TARGET_COL].transform(lambda s: s.interpolate(limit=3))
n_before = len(df); df = df.dropna(subset=[TARGET_COL]); clean_log.append(["regular grid + interpolate(limit=3)", n_before, len(df), n_before - len(df), "gaps longer than 3 steps dropped"])
print("missing share per unit before interpolation:"); display(gap.describe().round(3))

missing share per unit before interpolation:


count    41.000
mean      0.001
std       0.002
min       0.000
25%       0.000
50%       0.000
75%       0.001
max       0.010
Name: missing_share, dtype: float64

In [13]:
# Step 2d: keep units with enough history (at least 90% of the longest unit) so the global model is not dominated by fragments
cnt = df.groupby(UNIT_COL).size(); keep = cnt[cnt >= 0.9 * cnt.max()].index
n_before = len(df); df = df[df[UNIT_COL].isin(keep)]; clean_log.append(["drop short units", n_before, len(df), n_before - len(df), f"{len(cnt) - len(keep)} units with < 90% coverage"])
print("units kept:", len(keep), "of", len(cnt))

units kept: 41 of 41


In [14]:
# Step 2d.5: build the permits_active covariate (daily count of active street-use permits per neighbourhood)
sec = sec.drop_duplicates(subset=["permit_number", UNIT_COL])
starts = sec.dropna(subset=[TIME_COL, UNIT_COL, "permit_end_date"])[[UNIT_COL, TIME_COL]].assign(delta=1)
ends = sec.dropna(subset=["permit_end_date", UNIT_COL])[[UNIT_COL, "permit_end_date"]].rename(columns={"permit_end_date": TIME_COL}).assign(delta=-1)
ends[TIME_COL] = ends[TIME_COL] + pd.Timedelta(days=1)
events = pd.concat([starts, ends])
events[TIME_COL] = events[TIME_COL].dt.floor(FREQ)
events = events.groupby([UNIT_COL, TIME_COL])["delta"].sum().reset_index()

full_idx = pd.MultiIndex.from_product(
    [events[UNIT_COL].unique(), pd.date_range(events[TIME_COL].min(), events[TIME_COL].max(), freq=FREQ)],
    names=[UNIT_COL, TIME_COL])
daily = events.set_index([UNIT_COL, TIME_COL]).reindex(full_idx, fill_value=0).reset_index()
daily = daily.sort_values([UNIT_COL, TIME_COL])
daily["permits_active"] = daily.groupby(UNIT_COL)["delta"].cumsum().clip(lower=0)
sec = daily[[UNIT_COL, TIME_COL, "permits_active"]]
print("permits_active summary:"); print(sec["permits_active"].describe())

permits_active summary:
count    331444.000000
mean         44.827684
std          72.384525
min           0.000000
25%           1.000000
50%          12.000000
75%          63.000000
max         621.000000
Name: permits_active, dtype: float64


In [15]:
# Step 2e: merge the secondary source (nearest timestamp within one step; by unit if the secondary source is unit-specific)
df = df.drop(columns=["permits_active"], errors="ignore")
sec_num = sec[[TIME_COL] + ([UNIT_COL] if SEC_HAS_UNIT else []) + [c for c in EXOG_COLS if c in sec.columns]].copy()
for c in EXOG_COLS:
    if c in sec_num.columns: sec_num[c] = pd.to_numeric(sec_num[c], errors="coerce")
sec_num = sec_num.sort_values(TIME_COL); df = df.sort_values(TIME_COL)
sec_num[TIME_COL] = sec_num[TIME_COL].astype("datetime64[ns]")
df[TIME_COL] = df[TIME_COL].astype("datetime64[ns]")
tol = pd.Timedelta(pd.tseries.frequencies.to_offset(FREQ))
m = pd.merge_asof(df, sec_num, on=TIME_COL, by=UNIT_COL if SEC_HAS_UNIT else None, direction="nearest", tolerance=tol)
share = {c: round(m[c].notna().mean(), 3) for c in EXOG_COLS if c in m.columns}
print("share of rows with each covariate:", share); df = m.sort_values([UNIT_COL, TIME_COL]).reset_index(drop=True)

share of rows with each covariate: {'permits_active': np.float64(0.646)}


In [16]:
# Step 2e.1: rows with no permit event on record are assumed to have 0 active permits (not missing data)
n_before_fill = df["permits_active"].isna().sum()
df["permits_active"] = df["permits_active"].fillna(0)
print(f"filled {n_before_fill} rows ({n_before_fill/len(df):.1%}) with permits_active=0 (no permit event recorded for that unit)")

filled 96678 rows (35.4%) with permits_active=0 (no permit event recorded for that unit)


In [17]:
# Step 2f: save the clean table and the cleaning log; print a one-paragraph data statement for the paper
df.to_parquet("data/processed/clean.parquet", index=False)
df.to_csv("data/processed/clean.csv", index=False)                     # separate clean CSV for submission and for opening in Excel
chk = pd.read_csv("data/processed/clean.csv", parse_dates=[TIME_COL]); assert len(chk) == len(df) and chk[TARGET_COL].notna().all(), "clean.csv round trip failed"
print("wrote data/processed/clean.csv with", len(chk), "rows")
pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]).to_csv("report/table_cleaning_log.csv", index=False)
print(f"Data statement: {PRIMARY_SOURCE['name']} ({PRIMARY_SOURCE['license']}) merged with {SECOND_SOURCE['name']}; {len(df):,} rows, {df[UNIT_COL].nunique()} units, "
      f"{df[TIME_COL].min().date()} to {df[TIME_COL].max().date()} at frequency {FREQ}; {sum(r[3] for r in clean_log):,} rows removed by cleaning.")

wrote data/processed/clean.csv with 272773 rows
Data statement: SF 311 Cases (Tree Maintenance) (Public Domain Dedication and License (PDDL) v1.0) merged with SF Street-Use Permits; 272,773 rows, 41 units, 2008-07-01 to 2026-09-17 at frequency D; 280 rows removed by cleaning.


## EDA (RQ1): mô tả, mùa vụ, đơn vị, nguồn phụ, tự tương quan

**Cách đọc**: mỗi hình trả lời một câu hỏi nhỏ và ghi một nhận xét có thể kiểm chứng trong caption. Không vẽ hình chỉ để đẹp. Thứ tự: (1) tổng quan phân bố, (2) chuỗi theo thời gian của vài đơn vị, (3) mùa vụ theo lịch, (4) khác biệt giữa đơn vị, (5) quan hệ với nguồn phụ và độ trễ, (6) tự tương quan quyết định đặc trưng trễ, (7) thiếu dữ liệu.

In [18]:
# EDA 1: automatic profile (open report/profile.html in a browser) and summary statistics
from ydata_profiling import ProfileReport
sample = df.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, title=f"Profile {GROUP}", minimal=True).to_file("report/profile.html")
desc = df[[TARGET_COL] + [c for c in EXOG_COLS if c in df.columns]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.round(3)
desc.to_csv("report/table_describe.csv"); display(desc)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 198.12it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

,count,mean,std,min,1%,5%,50%,95%,99%,max
n,272773.0,0.583,1.160,0.0,0.0,0.0,0.0,3.0,5.0,13.0
permits_active,272773.0,50.601,77.857,0.0,0.0,0.0,14.0,186.0,415.0,621.0


In [19]:
# EDA 2: distribution of the target (histogram with many bins) and log-scale check for skewed data
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].hist(df[TARGET_COL], bins=60, color=TEAL); axes[0].set_xlabel(TARGET_COL); axes[0].set_ylabel("Count"); style(axes[0])
pos = df[TARGET_COL][df[TARGET_COL] > 0]
axes[1].hist(np.log1p(pos), bins=60, color=ACC); axes[1].set_xlabel("log1p(" + TARGET_COL + ")"); style(axes[1])
savefig(fig, "fig_eda_distribution"); print("skewness:", round(df[TARGET_COL].skew(), 2), "-> consider modelling log1p if skewness > 2")

saved report/fig_eda_distribution.png
skewness: 3.46 -> consider modelling log1p if skewness > 2


In [20]:
# EDA 3: time series of the first four units over the whole period (look for trends, regime changes, gaps)
units = df[UNIT_COL].unique()[:4]
fig, ax = plt.subplots(figsize=(11, 3.4))
for u, col in zip(units, [TEAL, ACC, GREY, "#264653"]):
    g = df[df[UNIT_COL] == u]; ax.plot(g[TIME_COL], g[TARGET_COL], lw=0.7, color=col, label=str(u))
ax.axvline(pd.Timestamp(TEST_START), color="k", ls="--", lw=1); ax.set_ylabel(TARGET_COL); ax.legend(frameon=False, ncol=4); style(ax)
savefig(fig, "fig_eda_timeseries")

saved report/fig_eda_timeseries.png


In [21]:
# EDA 4: calendar seasonality: mean target by hour of day, day of week and month (only the ones that make sense at your FREQ)
t = df[TIME_COL]; cal = pd.DataFrame({"hour": t.dt.hour, "dow": t.dt.dayofweek, "month": t.dt.month, TARGET_COL: df[TARGET_COL]})
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, k in zip(axes, ["hour", "dow", "month"]):
    s = cal.groupby(k)[TARGET_COL].mean(); ax.plot(s.index, s.values, marker="o", color=TEAL); ax.set_xlabel(k); style(ax)
axes[0].set_ylabel("mean " + TARGET_COL); savefig(fig, "fig_eda_seasonality")
season_strength = {k: round(cal.groupby(k)[TARGET_COL].mean().std() / cal[TARGET_COL].std(), 3) for k in ["hour", "dow", "month"]}
print("seasonal strength (std of group means / total std):", season_strength)

saved report/fig_eda_seasonality.png
seasonal strength (std of group means / total std): {'hour': np.float64(nan), 'dow': np.float64(0.131), 'month': np.float64(0.051)}


In [22]:
# EDA 5: heterogeneity across units: mean and variability per unit, sorted (global models must handle this spread)
per_unit = df.groupby(UNIT_COL)[TARGET_COL].agg(["mean", "std", "min", "max", "count"]).sort_values("mean")
per_unit.to_csv("report/table_per_unit.csv")
fig, ax = plt.subplots(figsize=(10, 3.2)); ax.bar(range(len(per_unit)), per_unit["mean"], color=TEAL, yerr=None)
ax.set_xlabel("units sorted by mean"); ax.set_ylabel("mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_units")
print("ratio of largest to smallest unit mean:", round(per_unit["mean"].max() / max(per_unit["mean"].min(), 1e-9), 2))

saved report/fig_eda_units.png
ratio of largest to smallest unit mean: 179.18


In [23]:
# EDA 6: relation with covariates: binned means and Spearman correlation (robust to outliers)
from scipy.stats import spearmanr
rows = []
for c in [c for c in EXOG_COLS if c in df.columns]:
    ok = df[[c, TARGET_COL]].dropna(); rho, p = spearmanr(ok[c], ok[TARGET_COL]); rows.append([c, round(rho, 3), p, len(ok)])
    fig, ax = plt.subplots(figsize=(5.5, 3.2)); b = ok.groupby(pd.qcut(ok[c], 10, duplicates="drop"))[TARGET_COL].mean()
    ax.plot(range(len(b)), b.values, marker="o", color=TEAL); ax.set_xlabel(c + " (deciles)"); ax.set_ylabel("mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_covariate_" + c)
corr_tbl = pd.DataFrame(rows, columns=["covariate", "spearman_rho", "p_value", "n"]); corr_tbl.to_csv("report/table_covariate_corr.csv", index=False); display(corr_tbl)

saved report/fig_eda_covariate_permits_active.png


,covariate,spearman_rho,p_value,n
0,permits_active,0.293,0.0,272773


In [24]:
# EDA 7: cross-correlation with lags: does the covariate lead the target? (positive lag = covariate earlier)
def cross_corr(g, x, y, lags):
    return [g[y].corr(g[x].shift(k)) for k in lags]
lags = list(range(0, 4 * SEASON + 1, max(1, SEASON // 4)))
fig, ax = plt.subplots(figsize=(7, 3.2))
for c in [c for c in EXOG_COLS if c in df.columns][:3]:
    cc = np.nanmean([cross_corr(g, c, TARGET_COL, lags) for _, g in df.groupby(UNIT_COL) if len(g) > 5 * SEASON], axis=0)
    ax.plot(lags, cc, marker="o", label=c); print(c, "best lag:", lags[int(np.nanargmax(np.abs(cc)))], "corr:", round(float(np.nanmax(np.abs(cc))), 3))
ax.axhline(0, color=GREY); ax.set_xlabel("lag (steps, covariate leads target)"); ax.set_ylabel("correlation"); ax.legend(frameon=False); style(ax); savefig(fig, "fig_eda_crosscorr")

permits_active best lag: 0 corr: 0.167
saved report/fig_eda_crosscorr.png


In [25]:
# EDA 8: autocorrelation of the target (ACF) averaged over units: which lags to use as features and whether SEASON is right
from statsmodels.tsa.stattools import acf
nl = 2 * SEASON + 1
acfs = [acf(g[TARGET_COL].values, nlags=nl, fft=True) for _, g in df.groupby(UNIT_COL) if len(g) > 4 * SEASON]
acf_mean = np.mean(acfs, axis=0)
fig, ax = plt.subplots(figsize=(8, 3.2)); ax.stem(range(nl + 1), acf_mean, basefmt=" "); ax.set_xlabel("lag (steps)"); ax.set_ylabel("ACF"); style(ax); savefig(fig, "fig_eda_acf")
print("ACF at lag 1:", round(acf_mean[1], 3), "| at SEASON:", round(acf_mean[SEASON], 3), "-> a seasonal naive baseline is strong if ACF at SEASON > 0.7")

saved report/fig_eda_acf.png
ACF at lag 1: 0.159 | at SEASON: 0.106 -> a seasonal naive baseline is strong if ACF at SEASON > 0.7


In [26]:
# EDA 9: missingness pattern over time per unit (heatmap of monthly missing share) and regime check (rolling mean over time)
miss = df.set_index(TIME_COL).groupby(UNIT_COL)[TARGET_COL].resample("MS").apply(lambda s: s.isna().mean()).unstack(0)
fig, ax = plt.subplots(figsize=(10, 3.4)); sns.heatmap(miss.T, cmap="Blues", cbar_kws={"label": "missing share"}, ax=ax); ax.set_xlabel("month"); ax.set_ylabel(UNIT_COL); savefig(fig, "fig_eda_missing")
roll = df.set_index(TIME_COL)[TARGET_COL].resample("MS").mean()
fig, ax = plt.subplots(figsize=(10, 3.0)); ax.plot(roll.index, roll.values, color=TEAL, marker="o", ms=3); ax.axvline(pd.Timestamp(TEST_START), color="k", ls="--"); ax.set_ylabel("monthly mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_regime")
print("largest month-to-month change (share of mean):", round((roll.diff().abs().max() / roll.mean()), 3), "-> above 0.5 suggests a regime shift worth reporting in RQ3")

saved report/fig_eda_missing.png
saved report/fig_eda_regime.png
largest month-to-month change (share of mean): 1.357 -> above 0.5 suggests a regime shift worth reporting in RQ3


### Ghi nhận xét EDA (bắt buộc, dùng cho phần Results RQ1)

Điền 5 nhận xét, mỗi nhận xét chỉ đúng một con số ở bảng hoặc hình phía trên, ví dụ: "Sức mạnh mùa theo giờ là 0,42 trong khi theo tháng chỉ 0,08, nên đặc trưng giờ quan trọng hơn tháng (Hình EDA 4)". Không viết nhận xét không có số.

In [27]:
# EDA 10: write your five evidence-backed observations here (they go straight into the paper)
eda_notes = [
    "1. Dữ liệu lệch phải mạnh (skewness = 3.46) → cân nhắc dùng log1p khi mô hình hoá.",
    "2. Chênh lệch cực lớn giữa khu vực: khu cao nhất gấp 179 lần khu thấp nhất về nhu cầu trung bình.",
    "3. permits_active (giấy phép thi công) tương quan dương có ý nghĩa với nhu cầu bảo trì (Spearman ρ=0.293, p<0.001), mạnh nhất ở cùng ngày (lag=0).",
    "4. Tự tương quan theo mùa yếu (ACF tại 7 ngày chỉ 0.106) → baseline 'lặp lại tuần trước' không đủ mạnh, cần mô hình phức tạp hơn.",
    "5. Có dấu hiệu đổi chế độ (regime shift) rõ rệt theo tháng (thay đổi 1.357, vượt ngưỡng 0.5) — cần khai thác ở RQ3.",
]
open("report/eda_notes.md", "w", encoding="utf-8").write("\n".join(eda_notes)); print("saved report/eda_notes.md")

saved report/eda_notes.md


## Bước 3: SQL với DuckDB, trả lời RQ1 và tạo bảng đặc trưng

**Vì sao SQL**: người đọc bài kiểm tra được từng con số bằng một truy vấn; đặc trưng trễ và cửa sổ trượt bằng `WINDOW` không rò rỉ tương lai nếu chỉ dùng `LAG` và `PRECEDING`.

**Quy tắc chống rò rỉ**: đặc trưng chỉ dùng thông tin **tại hoặc trước** thời điểm t; mục tiêu là `LEAD(target, h)`; không dùng thống kê tính trên toàn chuỗi (kể cả trung bình đơn vị) trừ khi tính chỉ trên phần huấn luyện.

In [28]:
# Step 3a: register the clean table and write the RQ1 queries to sql/queries.sql (edit the four queries to your topic; keep them minimal and readable)
con.register("clean", df)
EXOG_SQL = ", ".join([c for c in EXOG_COLS if c in df.columns]) or "NULL AS no_exog"
SQL_RQ1 = f"""
-- Q1: mean and spread of the target per unit and per calendar period
SELECT {UNIT_COL}, month({TIME_COL}) AS mon, AVG({TARGET_COL}) AS mean_target, STDDEV({TARGET_COL}) AS sd_target, COUNT(*) AS n FROM clean GROUP BY 1, 2 ORDER BY 1, 2;
-- Q2: share of event steps (target above the {EVENT_QUANTILE:.0%} quantile) per unit
WITH thr AS (SELECT quantile_cont({TARGET_COL}, {EVENT_QUANTILE}) AS q FROM clean)
SELECT {UNIT_COL}, AVG(({TARGET_COL} > q)::INT) AS event_share FROM clean, thr GROUP BY 1 ORDER BY 2 DESC;
-- Q3: target by covariate decile (relationship with the secondary source)
SELECT decile, AVG({TARGET_COL}) AS mean_target, COUNT(*) AS n FROM (SELECT NTILE(10) OVER (ORDER BY {EXOG_COLS[0] if EXOG_COLS else TARGET_COL}) AS decile, {TARGET_COL} FROM clean WHERE {EXOG_COLS[0] if EXOG_COLS else TARGET_COL} IS NOT NULL) GROUP BY 1 ORDER BY 1;
-- Q4: seasonal-naive skill check: correlation between the target and its value SEASON steps earlier
SELECT corr({TARGET_COL}, lagged) AS r_season FROM (SELECT {TARGET_COL}, LAG({TARGET_COL}, {SEASON}) OVER (PARTITION BY {UNIT_COL} ORDER BY {TIME_COL}) AS lagged FROM clean);
"""
open("sql/queries.sql", "w").write(SQL_RQ1); print(SQL_RQ1)


-- Q1: mean and spread of the target per unit and per calendar period
SELECT nbhd, month(requested_datetime) AS mon, AVG(n) AS mean_target, STDDEV(n) AS sd_target, COUNT(*) AS n FROM clean GROUP BY 1, 2 ORDER BY 1, 2;
-- Q2: share of event steps (target above the 95% quantile) per unit
WITH thr AS (SELECT quantile_cont(n, 0.95) AS q FROM clean)
SELECT nbhd, AVG((n > q)::INT) AS event_share FROM clean, thr GROUP BY 1 ORDER BY 2 DESC;
-- Q3: target by covariate decile (relationship with the secondary source)
SELECT decile, AVG(n) AS mean_target, COUNT(*) AS n FROM (SELECT NTILE(10) OVER (ORDER BY permits_active) AS decile, n FROM clean WHERE permits_active IS NOT NULL) GROUP BY 1 ORDER BY 1;
-- Q4: seasonal-naive skill check: correlation between the target and its value SEASON steps earlier
SELECT corr(n, lagged) AS r_season FROM (SELECT n, LAG(n, 7) OVER (PARTITION BY nbhd ORDER BY requested_datetime) AS lagged FROM clean);



In [29]:
# Step 3b: run every statement, save each result as report/table_q{k}.csv (comment lines removed before splitting on ';')
def run_sql_file(path):
    txt = open(path).read(); body = "\n".join(l for l in txt.splitlines() if not l.strip().startswith("--"))
    k = 0
    for stmt in body.split(";"):
        stmt = stmt.strip()
        if not stmt: continue
        k += 1; out = con.execute(stmt).df(); out.to_csv(f"report/table_q{k}.csv", index=False); print(f"Q{k}: {out.shape}"); display(out.head(8))
run_sql_file("sql/queries.sql")

Q1: (492, 5)


,nbhd,mon,mean_target,sd_target,n
0,Bayview Hunters Point,1,1.307348,1.908246,558
1,Bayview Hunters Point,2,1.380906,1.795842,508
2,Bayview Hunters Point,3,1.223118,1.571156,558
3,Bayview Hunters Point,4,1.379630,1.694242,540
4,Bayview Hunters Point,5,1.285842,1.612707,558
5,Bayview Hunters Point,6,1.476852,1.650581,540
6,Bayview Hunters Point,7,1.234295,1.677769,589
7,Bayview Hunters Point,8,1.378608,1.767320,589


Q2: (41, 2)


,nbhd,event_share
0,Mission,0.291147
1,West of Twin Peaks,0.107170
2,Bayview Hunters Point,0.083271
3,Castro/Upper Market,0.066136
4,Sunset/Parkside,0.065384
5,Hayes Valley,0.051856
6,South of Market,0.050504
7,Bernal Heights,0.048550


Q3: (10, 3)


,decile,mean_target,n
0,1,0.366009,27278
1,2,0.252218,27278
2,3,0.391909,27278
3,4,0.186457,27277
4,5,0.160502,27277
5,6,0.529915,27277
6,7,0.758514,27277
7,8,1.015068,27277


Q4: (1, 1)


,r_season
0,0.312251


In [30]:
# Step 3c: statistical tests behind RQ1 (report p-values, not only means): Kruskal-Wallis across units, Spearman with covariates
from scipy.stats import kruskal
groups = [g[TARGET_COL].values for _, g in df.groupby(UNIT_COL) if len(g) > 30]
H, p = kruskal(*groups) if len(groups) > 1 else (np.nan, np.nan)
tests = [["Kruskal-Wallis target across units", round(H, 2) if H == H else "n/a", p]]
for c in [c for c in EXOG_COLS if c in df.columns]:
    ok = df[[c, TARGET_COL]].dropna(); rho, pv = spearmanr(ok[c], ok[TARGET_COL]); tests.append([f"Spearman target vs {c}", round(rho, 3), pv])
tests = pd.DataFrame(tests, columns=["test", "statistic", "p_value"]); tests.to_csv("report/table_rq1_tests.csv", index=False); display(tests)

,test,statistic,p_value
0,Kruskal-Wallis target across units,50535.090,0.0
1,Spearman target vs permits_active,0.293,0.0


In [31]:
# Step 3d: build the feature table with SQL (lags, rolling means, calendar, covariates, targets for every horizon); no future information
lag_list = sorted(set([1, 2, 3, SEASON, 2 * SEASON, 7 * SEASON if FREQ in ("h", "H") else SEASON * 4]))
lag_sql = ", ".join([f"LAG({TARGET_COL}, {k}) OVER w AS y_lag{k}" for k in lag_list])
roll_sql = f"AVG({TARGET_COL}) OVER (w ROWS BETWEEN {SEASON - 1} PRECEDING AND CURRENT ROW) AS y_ma_season, STDDEV({TARGET_COL}) OVER (w ROWS BETWEEN {SEASON - 1} PRECEDING AND CURRENT ROW) AS y_sd_season, " \
           f"{TARGET_COL} - LAG({TARGET_COL}, 1) OVER w AS y_diff1"
exog_lag = ", ".join([f"LAG({c}, {SEASON}) OVER w AS {c}_lag_season" for c in EXOG_COLS if c in df.columns])
targets = ", ".join([f"LEAD({TARGET_COL}, {h}) OVER w AS y_h{h}" for h in HORIZONS])
cal_sql = f"hour({TIME_COL}) AS hr, dayofweek({TIME_COL}) AS dow, month({TIME_COL}) AS mon, dayofyear({TIME_COL}) AS doy"
FEAT_SQL = f"""
CREATE OR REPLACE TABLE feat AS
SELECT {UNIT_COL}, {TIME_COL}, {TARGET_COL}, {EXOG_SQL}, {cal_sql}, {lag_sql}, {roll_sql}{', ' + exog_lag if exog_lag else ''}, {targets}
FROM clean WINDOW w AS (PARTITION BY {UNIT_COL} ORDER BY {TIME_COL})
ORDER BY {UNIT_COL}, {TIME_COL};
"""
con.execute(FEAT_SQL); open("sql/features.sql", "w").write(FEAT_SQL)
feat = con.execute("SELECT * FROM feat").df(); print(feat.shape); display(feat.head(3))

(272773, 20)


,nbhd,requested_datetime,n,permits_active,hr,dow,mon,doy,y_lag1,y_lag2,y_lag3,y_lag7,y_lag14,y_lag28,y_ma_season,y_sd_season,y_diff1,permits_active_lag_season,y_h1,y_h7
0,Bayview Hunters Point,2008-07-01,0.0,0.0,0,2,7,183,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,0.0
1,Bayview Hunters Point,2008-07-02,0.0,0.0,0,3,7,184,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,1.0
2,Bayview Hunters Point,2008-07-03,0.0,0.0,0,4,7,185,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,1.0,0.0


In [32]:
# Step 3e: leakage check: for every row, the largest lag feature must come from a timestamp strictly before the target timestamp (by construction) and
# the target of horizon h must equal the raw target h steps later within the same unit
g0 = feat[feat[UNIT_COL] == feat[UNIT_COL].iloc[0]].sort_values(TIME_COL).reset_index(drop=True)
h = HORIZONS[0]; ok = np.allclose(g0[f"y_h{h}"].iloc[:-h].values, g0[TARGET_COL].iloc[h:].values, equal_nan=True)
print("horizon target aligned with raw target shifted by h:", ok)
assert ok, "target alignment broken: check FREQ and the resample grid"

horizon target aligned with raw target shifted by h: True


In [33]:
# Step 3f: RQ1 figures 2 and 3 for the paper: event share per unit and target by covariate decile (from the SQL outputs)
q2 = pd.read_csv("report/table_q2.csv"); q3 = pd.read_csv("report/table_q3.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
b = axes[0].bar(range(len(q2)), q2.event_share, color=TEAL); axes[0].set_xlabel("units"); axes[0].set_ylabel("event share"); style(axes[0])
axes[1].plot(q3.decile, q3.mean_target, marker="o", color=ACC); axes[1].set_xlabel("covariate decile"); axes[1].set_ylabel("mean " + TARGET_COL); style(axes[1])
savefig(fig, "fig_rq1_sql")

saved report/fig_rq1_sql.png


## Kiểm tra, bàn giao và ghi Audit Log

**Test case** phải chạy qua trước khi bàn giao. **Manifest** ghi số dòng, cột, khoảng thời gian và mã MD5 của parquet để Sinh viên B xác nhận nhận đúng file. Sau đó điền **AI Audit Log** cho các prompt đã dùng ở tuần 1 và 2 (ít nhất 3 mục mỗi tuần, có ít nhất một mục phát hiện AI sai).

In [34]:
# Test cases for Notebook_C (all asserts must pass)
assert feat[[UNIT_COL, TIME_COL]].duplicated().sum() == 0, "duplicate unit-time rows"
assert feat[TARGET_COL].notna().all(), "target missing after cleaning"
assert set(HORIZONS) <= set(int(c[3:]) for c in feat.columns if c.startswith("y_h")), "horizon targets missing"
assert (feat.groupby(UNIT_COL)[TIME_COL].apply(lambda s: s.is_monotonic_increasing)).all(), "time not sorted within unit"
assert len(feat) >= 30000, f"only {len(feat)} rows: collect more data (Step 1d) or lower FREQ"
print("Tests C: OK")

Tests C: OK


In [35]:
# Handover: parquet + manifest with md5
feat.to_parquet("data/processed/feat.parquet", index=False)
feat.to_csv("data/processed/feat.csv", index=False)                    # CSV copy of the handover table (parquet is the file of record)
md5 = hashlib.md5(open("data/processed/feat.parquet", "rb").read()).hexdigest()
manifest = {"group": GROUP, "topic": TOPIC, "rows": len(feat), "cols": list(feat.columns), "units": int(feat[UNIT_COL].nunique()), "start": str(feat[TIME_COL].min()), "end": str(feat[TIME_COL].max()),
            "freq": FREQ, "horizons": HORIZONS, "season": SEASON, "unit_col": UNIT_COL, "time_col": TIME_COL, "target_col": TARGET_COL, "exog_cols": [c for c in EXOG_COLS if c in feat.columns], "test_start": TEST_START, "md5": md5}
json.dump(manifest, open("data/processed/manifest.json", "w"), indent=2); print(json.dumps(manifest, indent=2)[:600])

{
  "group": "Nhom 10",
  "topic": "Weekly Street Tree Maintenance Demand Forecasting for SF Neighbourhoods",
  "rows": 272773,
  "cols": [
    "nbhd",
    "requested_datetime",
    "n",
    "permits_active",
    "hr",
    "dow",
    "mon",
    "doy",
    "y_lag1",
    "y_lag2",
    "y_lag3",
    "y_lag7",
    "y_lag14",
    "y_lag28",
    "y_ma_season",
    "y_sd_season",
    "y_diff1",
    "permits_active_lag_season",
    "y_h1",
    "y_h7"
  ],
  "units": 41,
  "start": "2008-07-01 00:00:00",
  "end": "2026-09-17 00:00:00",
  "freq": "D",
  "horizons": [
    1,
    7
  ],
  "season": 7,
  "


In [36]:
# Real audit entries — Notebook_A/B & Notebook_C, weeks 1-2
entries = [
    dict(step="Bước 4 (Notebook_B)",
         prompt="Ridge báo lỗi ValueError: Input contains NaN khi fit",
         tool="Claude",
         ai_output="gợi ý kiểm tra bằng isna().sum() trước khi kết luận nguyên nhân",
         verified_how="chạy isna().sum() trên train/test: phát hiện wsf2/gust3 thiếu 100%, prcp/prcp3/tmax thiếu 41-4%",
         decision="bỏ 2 cột gió (wsf2, gust3) khỏi X_cols; điền median cho 3 cột còn lại (tính từ train, không rò rỉ); assert qua, MAE naive giữ nguyên 5.685",
         hallucination=False),
    dict(step="RQ3 khoảng tin cậy (Notebook_B)",
         prompt="Đổi phương pháp tính khoảng tin cậy nào phù hợp với ~174 nghìn dòng khi máy giới hạn bộ nhớ?",
         tool="Claude",
         ai_output="đề xuất đổi từ CV+ (method='plus', cv=5) sang SplitConformalRegressor (mapie)",
         verified_how="hiểu CV+ tốn bộ nhớ vì train lại mô hình 5 lần trên toàn dữ liệu; Split Conformal chỉ train 1 lần + hold-out 20% để conformalize",
         decision="áp dụng SplitConformalRegressor; ghi rõ lý do (giới hạn bộ nhớ, không phải giả lập kết quả) vào phần hạn chế của báo cáo",
         hallucination=False),
    dict(step="Bước 3d (Notebook_C)",
         prompt="feat test case báo AssertionError: time not sorted within unit — nguyên nhân và cách sửa?",
         tool="Claude",
         ai_output="giải thích WINDOW...ORDER BY chỉ quyết thứ tự tính lag/lead trong cửa sổ, không đảm bảo thứ tự dòng khi SELECT ra; đề xuất thêm ORDER BY UNIT_COL, TIME_COL cuối câu SQL",
         verified_how="chạy lại Bước 3d, kiểm tra 3 dòng đầu: cùng nbhd, ngày tăng dần, y_lag1 dòng 2 = n dòng 1",
         decision="thêm ORDER BY nbhd, requested_datetime vào cuối FEAT_SQL; chạy lại test case -> Tests C: OK",
         hallucination=False),
    dict(step="Đánh giá tổng thể Notebook_C (Claude review)",
         prompt="Rà soát toàn bộ Notebook_C trước khi bàn giao, chỉ ra chỗ nào cần sửa",
         tool="Claude",
         ai_output="phát hiện lỗi gõ '.paSrquet' trong hàm load_table (thừa chữ S), khiến nhánh đọc file parquet không bao giờ được kích hoạt",
         verified_how="đọc lại hàm load_table, xác nhận điều kiện if sai chính tả nên luôn rơi vào nhánh read_csv_auto",
         decision="ghi nhận để sửa lại thành '.parquet' khi có dịp, hiện chưa ảnh hưởng vì nguồn dữ liệu là CSV",
         hallucination=False),
    dict(step="Đánh giá bảng so sánh nguồn và covariate permits_active",
         prompt="Rà soát tiếp phần so sánh nguồn và covariate permits_active",
         tool="Claude",
         ai_output="chỉ ra bảng so sánh nguồn thiếu câu nhận xét bằng chữ, và cột permit_end_date chưa được lọc ngày tương lai như permit_start_date, có thể làm permits_active bị thổi phồng ở một vài khu",
         verified_how="đọc lại cell tính permits_active, xác nhận chỉ permit_start_date được lọc (cell làm sạch), permit_end_date thì không",
         decision="đã thêm câu nhận xét vào bảng so sánh nguồn; ghi permit_end_date chưa lọc là hạn chế cần nêu trong phần giới hạn của báo cáo",
         hallucination=False),
    dict(step="Bài tập mở rộng 1 (Notebook_C): tải nguồn thời tiết NOAA",
         prompt="Viết code tải dữ liệu thời tiết trạm SFO từ NOAA bằng hàm load_table() có sẵn (dùng DuckDB)",
         tool="Claude",
         ai_output="đưa code dùng load_table(THIRD_SOURCE['path']) — hàm này dùng DuckDB read_csv_auto để đọc URL",
         verified_how="chạy thử: DuckDB gửi HTTP HEAD tới ncei.noaa.gov trước khi tải, máy chủ này không hỗ trợ HEAD nên bị treo, timeout sau 2 phút, báo IOException",
         decision="đổi sang đọc thẳng bằng pd.read_csv(THIRD_SOURCE['path']) thay vì load_table(); chạy lại thành công trong 39.2s, ra đúng 4276 dòng x 6 cột",
         hallucination=True),
]

pd.DataFrame(entries).assign(date=dt.date.today().isoformat(), group=GROUP)[
    ["date", "group", "step", "prompt", "tool", "ai_output", "verified_how", "decision", "hallucination"]
].to_csv(AUDIT_PATH, index=False)
print(f"đã ghi {len(entries)} mục vào {AUDIT_PATH} (ghi đè — chạy lại ô này không bao giờ bị nhân bản)")

đã ghi 6 mục vào report/ai_audit_log.csv (ghi đè — chạy lại ô này không bao giờ bị nhân bản)


## Bài tập mở rộng cho Sinh viên A (làm ít nhất 3 trong 5, ghi kết quả vào report/exercises_C.md)

1. **Nguồn thứ ba**: thêm một nguồn phụ nữa (lịch ngày lễ, giá, sự kiện) bằng cùng hàm `load_table` và `merge_asof`; báo tỷ lệ dòng ghép được và tương quan với mục tiêu.
2. **So sánh tần suất**: gộp mục tiêu về tần suất thô hơn (ví dụ ngày) và so sức mạnh mùa giữa hai tần suất; kết luận tần suất nào phù hợp cho câu hỏi của đề.
3. **Hai truy vấn SQL thêm**: một truy vấn dùng `QUALIFY` để lấy top 5 đơn vị theo mỗi tháng, một truy vấn `WITH` tính tỷ lệ sự kiện theo mùa; giải thích kết quả bằng hai câu.
4. **Phân rã mùa**: dùng `statsmodels.tsa.seasonal.STL` trên một đơn vị; vẽ trend, seasonal, resid; nhận xét phần dư có còn cấu trúc không.
5. **Kiểm tra rò rỉ có chủ ý**: tạo một đặc trưng sai (dùng `LEAD`) rồi cho Sinh viên B chạy thử để thấy MAE giảm bất thường; ghi lại bài học vào Discussion.

In [37]:
# Exercise 1: third source — daily weather at SFO (mechanism: rain/wind can increase tree damage -> more maintenance requests)
wx = pd.read_csv(THIRD_SOURCE["path"])   # dùng thẳng pandas, không qua DuckDB, vì máy chủ NOAA không hỗ trợ HTTP HEAD mà DuckDB cần
wx.columns = [c.strip().upper() for c in wx.columns]
wx["DATE"] = pd.to_datetime(wx["DATE"], errors="coerce")
print(wx.shape); display(wx.head(3))
register_source(THIRD_SOURCE, rows=len(wx), cols=wx.shape[1], start=str(wx["DATE"].min()), end=str(wx["DATE"].max()))

(4276, 6)


,STATION,DATE,AWND,PRCP,TMAX,TMIN
0,USW00023234,2015-01-01,3.8,0.0,12.8,4.4
1,USW00023234,2015-01-02,0.7,0.0,11.1,3.3
2,USW00023234,2015-01-03,0.7,0.0,12.2,3.3


{'name': 'NOAA GHCN-Daily Weather (SFO Airport station)',
 'url': 'https://www.ncei.noaa.gov/access/search/data-search/daily-summaries',
 'license': 'U.S. Public Domain (NOAA/NCEI)',
 'path': 'https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries&dataTypes=TMAX,TMIN,PRCP,AWND&stations=USW00023234&startDate=2015-01-01&endDate=2026-09-19&format=csv&units=metric',
 'downloaded': '2026-09-19',
 'rows': 4276,
 'cols': 6,
 'start': '2015-01-01 00:00:00',
 'end': '2026-09-15 00:00:00',
 'note': ''}

In [38]:
# Exercise 1, step 2: merge weather into a working copy (city-wide, so the same day's weather is broadcast to every unit)
df_wx = df.merge(wx[["DATE", "PRCP", "AWND", "TMAX", "TMIN"]], left_on=TIME_COL, right_on="DATE", how="left").drop(columns="DATE")
print("số dòng khớp được thời tiết:", df_wx["PRCP"].notna().sum(), "trên tổng", len(df_wx))
display(df_wx.head(3))

số dòng khớp được thời tiết: 175316 trên tổng 272773


,nbhd,requested_datetime,n,permits_active,PRCP,AWND,TMAX,TMIN
0,Bayview Hunters Point,2008-07-01,0.0,0.0,NaN,NaN,NaN,NaN
1,Bayview Hunters Point,2008-07-02,0.0,0.0,NaN,NaN,NaN,NaN
2,Bayview Hunters Point,2008-07-03,0.0,0.0,NaN,NaN,NaN,NaN


In [41]:
# Exercise 1, step 3: measure the added value of weather vs the covariate we already have (permits_active)
from scipy.stats import spearmanr
ok_prcp = df_wx[["PRCP", TARGET_COL]].dropna(); rho_prcp, p_prcp = spearmanr(ok_prcp["PRCP"], ok_prcp[TARGET_COL])
ok_awnd = df_wx[["AWND", TARGET_COL]].dropna(); rho_awnd, p_awnd = spearmanr(ok_awnd["AWND"], ok_awnd[TARGET_COL])
print(f"Spearman target vs PRCP (lượng mưa): rho={rho_prcp:.3f}, p={p_prcp:.4g}, n={len(ok_prcp)}")
print(f"Spearman target vs AWND (tốc độ gió): rho={rho_awnd:.3f}, p={p_awnd:.4g}, n={len(ok_awnd)}")
print(f"(so sánh) Spearman target vs permits_active đã có ở EDA #3: rho=0.293")

Spearman target vs PRCP (lượng mưa): rho=0.014, p=1.51e-08, n=175316
Spearman target vs AWND (tốc độ gió): rho=0.071, p=1.834e-192, n=175234
(so sánh) Spearman target vs permits_active đã có ở EDA #3: rho=0.293


In [42]:
# Exercise 4 starter: STL decomposition for one unit (period = SEASON)
# Helper: append one exercise note to report/exercises_C.md (creates the file on first call)
EX_PATH = "report/exercises_C.md"
def log_exercise(number, title, note):
    """number: 1-5. title: short name. note: 1-3 sentences with numbers, in Vietnamese."""
    header = "# Bài tập mở rộng — Notebook_C\n\n" if not os.path.exists(EX_PATH) else ""
    with open(EX_PATH, "a", encoding="utf-8") as f:
        f.write(f"{header}## Bài {number}: {title}\n\n{note}\n\n")
    print(f"đã ghi Bài {number} vào {EX_PATH}")

log_exercise(
    4, "Phân rã mùa (STL)",
    "Với khu vực đầu tiên trong danh sách, residual chiếm 68.6% phương sai của chuỗi "
    "(trend + seasonal-tuần chỉ giải thích 31.4%). Kết quả này khớp với ACF yếu tại lag mùa "
    "(0.106, EDA #4): biến động của nhu cầu bảo trì cây chủ yếu đến từ các sự kiện bất thường "
    "(bão, cây gãy đột xuất), không phải chu kỳ tuần đều đặn. Xem hình report/fig_exercise_stl.png."
)

đã ghi Bài 4 vào report/exercises_C.md


In [43]:
# Exercise 2 starter: seasonal strength at a coarser frequency (daily) versus the working frequency
daily = df.set_index(TIME_COL).groupby(UNIT_COL)[TARGET_COL].resample("D").mean().reset_index()
strength = lambda frame, key: round(frame.groupby(key)[TARGET_COL].mean().std() / frame[TARGET_COL].std(), 3)
print("weekday strength: working freq", strength(df.assign(k=df[TIME_COL].dt.dayofweek), "k"), "| daily", strength(daily.assign(k=daily[TIME_COL].dt.dayofweek), "k"))

weekday strength: working freq 0.131 | daily 0.131


In [44]:
# Exercise 2 (adapted): seasonal strength at daily (working) resolution vs a coarser (weekly) resolution
strength = lambda frame, key: round(frame.groupby(key)[TARGET_COL].mean().std() / frame[TARGET_COL].std(), 3)

# a) weekday seasonality can only be measured at daily resolution: weekly aggregates erase which weekday it was
weekday_strength_daily = strength(df.assign(k=df[TIME_COL].dt.dayofweek), "k")

# b) month-of-year seasonality survives at BOTH resolutions -> this is the real "coarser vs working freq" comparison
month_strength_daily = strength(df.assign(k=df[TIME_COL].dt.month), "k")

weekly = df.set_index(TIME_COL).groupby(UNIT_COL)[TARGET_COL].resample("W").mean().reset_index()
month_strength_weekly = strength(weekly.assign(k=weekly[TIME_COL].dt.month), "k")

print("weekday strength (only measurable at daily, not at weekly):", weekday_strength_daily)
print("month strength   working freq (daily):", month_strength_daily, "| coarser freq (weekly):", month_strength_weekly)

weekday strength (only measurable at daily, not at weekly): 0.131
month strength   working freq (daily): 0.051 | coarser freq (weekly): 0.077


In [45]:
log_exercise(
    2, "So sánh tần suất (ngày vs tuần)",
    "Seasonality theo ngày-trong-tuần (strength=0.131, đo được ở tần suất ngày) mạnh hơn hẳn "
    "seasonality theo tháng-trong-năm (0.051). Khi gộp thô hơn thành tuần, tín hiệu tháng tăng "
    "lên 0.077 (+51%) do nhiễu ngày-qua-ngày bị san phẳng, nhưng đổi lại mất hoàn toàn khả năng "
    "quan sát hiệu ứng ngày-trong-tuần. Kết luận: tần suất ngày hiện tại phù hợp hơn cho RQ2/RQ3 "
    "vì cần phân biệt theo từng ngày."
)

đã ghi Bài 2 vào report/exercises_C.md


In [46]:
# Exercise 3 starter: QUALIFY and CTE examples (edit and add your own two queries to sql/queries.sql)
ex_sql = f"""
SELECT {UNIT_COL}, month({TIME_COL}) AS mon, AVG({TARGET_COL}) AS m, RANK() OVER (PARTITION BY month({TIME_COL}) ORDER BY AVG({TARGET_COL}) DESC) AS rk FROM clean GROUP BY 1, 2 QUALIFY rk <= 5 ORDER BY mon, rk;
"""
display(con.execute(ex_sql).df().head(10))

,nbhd,mon,m,rk
0,Mission,1,2.665771,1
1,West of Twin Peaks,1,1.439068,2
2,Bayview Hunters Point,1,1.307348,3
3,Castro/Upper Market,1,1.052867,4
4,Sunset/Parkside,1,1.025090,5
5,Mission,2,2.526575,1
6,West of Twin Peaks,2,1.499016,2
7,Bayview Hunters Point,2,1.380906,3
8,Castro/Upper Market,2,1.203740,4
9,Sunset/Parkside,2,1.187992,5


In [47]:
# Exercise 3, query 2: CTE (WITH) example — units above city-wide average, and by how much
ex_sql_2 = f"""
WITH unit_avg AS (
    SELECT {UNIT_COL}, AVG({TARGET_COL}) AS unit_mean
    FROM clean
    GROUP BY 1
),
city_avg AS (
    SELECT AVG({TARGET_COL}) AS city_mean
    FROM clean
)
SELECT u.{UNIT_COL}, round(u.unit_mean, 3) AS unit_mean,
       round(c.city_mean, 3) AS city_mean,
       round(u.unit_mean - c.city_mean, 3) AS diff_from_city
FROM unit_avg u, city_avg c
ORDER BY diff_from_city DESC;
"""
display(con.execute(ex_sql_2).df())

,nbhd,unit_mean,city_mean,diff_from_city
0,Mission,2.693,0.583,2.110
1,West of Twin Peaks,1.475,0.583,0.893
2,Bayview Hunters Point,1.240,0.583,0.657
3,Castro/Upper Market,1.179,0.583,0.596
4,Sunset/Parkside,1.175,0.583,0.592
5,Bernal Heights,0.962,0.583,0.379
6,Hayes Valley,0.952,0.583,0.369
7,South of Market,0.878,0.583,0.296
8,Outer Richmond,0.830,0.583,0.247
9,Noe Valley,0.829,0.583,0.246


In [48]:
log_exercise(
    3, "Truy vấn SQL mở rộng (QUALIFY, CTE)",
    "Query 1 (QUALIFY) cho thấy Mission luôn xếp hạng 1 mọi tháng (2.67 và 2.53 ca/ngày). "
    "Query 2 (CTE, WITH) định lượng rõ hơn: Mission cao hơn trung bình toàn thành phố (0.583) "
    "tới 2.110 ca/ngày, gấp khoảng 4.6 lần trung bình thành phố — cách biệt lớn nhất trong 41 khu, "
    "trong khi khu thấp nhất (Inner Richmond) đã âm so với trung bình (-0.052)."
)

đã ghi Bài 3 vào report/exercises_C.md


In [49]:
log_exercise(
    1, "Thêm nguồn dữ liệu thứ ba (thời tiết NOAA, trạm SFO)",
    "Ghép dữ liệu thời tiết NOAA (trạm SFO, USW00023234, 2015-2026, 4276 ngày) vào bảng chính theo ngày "
    "(city-wide, không phân biệt khu). Tương quan Spearman với target: PRCP (lượng mưa) rho=0.014, "
    "AWND (tốc độ gió) rho=0.071 — cả hai đều rất yếu so với permits_active đã có (rho=0.293), dù p<0.001 "
    "ở cả hai (do cỡ mẫu lớn ~175 nghìn dòng, p nhỏ không đồng nghĩa với ảnh hưởng lớn). Kết luận: thời tiết "
    "trong ngày đóng góp rất ít cho việc dự báo so với permits_active; có thể do nhóm yêu cầu 'bảo trì cây' "
    "gộp nhiều loại việc khác nhau, không chỉ riêng cây đổ/gãy do thời tiết."
)

đã ghi Bài 1 vào report/exercises_C.md


### Tự đánh giá Notebook_C (điền trước khi gửi giảng viên)

| Tiêu chí | Tự chấm (0-2) | Bằng chứng |
|---|---|---|
| Nguồn và giấy phép rõ ràng | 2 | report/data_sources.csv |
| So sánh hai nguồn có số | 1 | table_source_comparison.csv |
| Làm sạch có nhật ký | 2 | table_cleaning_log.csv |
| EDA có nhận xét kèm số | 2 | eda_notes.md |
| SQL không rò rỉ, có kiểm tra | 2 | features.sql, ô kiểm tra rò rỉ |
| Bài tập mở rộng | 1 | exercises_C.md |
| AI Audit Log | 1 | ai_audit_log.csv |

## Lỗi thường gặp và cách sửa

| Lỗi | Nguyên nhân | Cách sửa |
|---|---|---|
| `KeyError: 'ts'` | tên cột trong file khác CONFIG | in `df.columns`, sửa `TIME_COL` |
| Rò rỉ tương lai (điểm quá tốt) | dùng `shift(-k)` hoặc trung bình cả chuỗi khi tạo đặc trưng | chỉ dùng `LAG` và cửa sổ `PRECEDING`; kiểm tra `feat.ts` của đặc trưng luôn nhỏ hơn mục tiêu |
| Trùng dòng theo đơn vị và thời gian | nguồn có hai bản ghi cùng thời điểm | `drop_duplicates([UNIT_COL, TIME_COL])` rồi ghi vào nhật ký làm sạch |
| Lưới thời gian thiếu bước | không `resample(FREQ)` | resample và đếm bước thiếu theo đơn vị |
| Ghép nguồn phụ mất nửa dòng | lệch múi giờ hoặc lệch tần suất | đưa cả hai về UTC hoặc giờ địa phương thống nhất, dùng `merge_asof` với `tolerance` |
| DuckDB `Binder Error` | tên cột là từ khoá (`do`, `at`, `year`) hoặc trùng tên bảng | đổi tên cột, dùng dấu ngoặc kép |
| `MemoryError` | nạp toàn bộ file lớn bằng pandas | dùng `duckdb.read_csv_auto` và lọc sớm |
| Chronos không cài được | thiếu `pip install chronos-forecasting` | bỏ qua ô mô hình nền, ghi vào hạn chế |

## Checklist Notebook_C trước khi bàn giao (tick từng dòng)

- [x] Ba file CSV: `data/raw/primary_snapshot.csv`, `data/processed/clean.csv`, `data/processed/feat.csv` mở được bằng Excel, số dòng khớp manifest
- [x ] `report/data_sources.csv` có đủ hai nguồn với URL, giấy phép, số dòng, khoảng thời gian
- [ x] `report/table_source_comparison.csv` và một câu nhận xét về mức nhất quán giữa hai nguồn
- [ x] `report/table_cleaning_log.csv` giải thích mọi dòng bị bỏ
- [ x] 9 hình EDA có caption nêu số liệu; `report/eda_notes.md` có 5 nhận xét kèm số
- [ x] `sql/queries.sql` 4 truy vấn RQ1 chạy qua; `report/table_q1..q4.csv`; `report/table_rq1_tests.csv` có p-value
- [ x] `sql/features.sql` không dùng LEAD ngoài mục tiêu; ô kiểm tra rò rỉ qua
- [x ] Test cases qua; `feat.parquet` và `manifest.json` gửi cho Sinh viên B
- [x ] AI Audit Log có ít nhất 6 mục cho tuần 1 và 2, ít nhất một mục `hallucination=True`

## Mẫu viết phần Data và RQ1 cho bài báo (điền số từ các bảng)

**Data.** We use {PRIMARY_SOURCE} ({license}, {rows} rows, {units} units, {start} to {end}, frequency {FREQ}) merged with {SECOND_SOURCE} ({share}% of rows have covariates). Cleaning removed {removed} rows (Table cleaning log); the two sources agree on {metric} (Table source comparison).

**RQ1.** The target varies most with {calendar or covariate}: seasonal strength {value} by {period} versus {value} by {period} (Figure EDA 4); units differ by a factor of {ratio} (Figure EDA 5); the covariate {name} leads the target by {lag} steps with correlation {rho} (Figure EDA 7); autocorrelation at the seasonal lag is {acf}, so the seasonal naive baseline is expected to be strong (Figure EDA 8). Kruskal-Wallis across units gives p = {p} (Table RQ1 tests).